# Hierarchical SAC — Forecast Tracking (Hier-2, FeatureEngineered)

Upper-level SAC coordinator (9-obs + 6h/12h look-ahead) on top of
frozen DecentralizedFESAC lower agents.

**Prerequisite:** `decentralized_sac_battery.ipynb` must have finished.

Kernel: miniconda3 (base) or data_methods

In [1]:
import subprocess, sys
for pkg in ['nrel-pysam', 'wandb', 'plotly', 'gymnasium', 'simplejson']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print('Dependencies ready.')


Dependencies ready.


## 1. Setup

In [2]:
import sys, os, json, copy, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
warnings.filterwarnings('ignore')
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_API_KEY"] = "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"

HERE = Path.cwd()
REPO_DIR = next(p for p in [HERE] + list(HERE.parents) if (p / 'citylearn').is_dir())
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'SERVER'))
sys.path.insert(0, str(REPO_DIR / 'harish_work'))

from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import BasicBatteryRBC
from citylearn.reward_function import RewardFunction
from citylearn.rl import PolicyNetwork, SoftQNetwork, ReplayBuffer
from feature_engineer import FeatureEngineer
from KPIs import compute_kpis
import wandb
import plotly.graph_objects as go

DATASET_DIR   = REPO_DIR / 'data' / 'datasets' / 'annex96_ce1_tx_neighborhood'
SCHEMA_PATH   = DATASET_DIR / 'schema.json'
# Local path avoids OneDrive sync conflicts during large torch.save() operations
MODELS_DIR    = Path(r'C:/Users/20213697/citylearn_models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TX_SIM_START   = 3624
TX_SIM_END     = 4343
N_EPISODES     = 100
TRACKING_SCALE = 1000.0
WANDB_ENTITY   = 'CityLearn-TeamB'
WANDB_PROJECT  = 'CityLearn'

district_target = pd.read_csv(DATASET_DIR / 'district_target.csv')[
    'district_load_target'].values[TX_SIM_START:TX_SIM_END + 1]
target_list = district_target.tolist()
print(f'District target: {len(district_target)} steps, mean={district_target.mean():.2f} kWh/h')

CONTROLLER_NAME = 'Hierarchical-Hier2-Forecast'


District target: 720 steps, mean=19.50 kWh/h


## 2. TrackingReward

In [3]:
class TrackingReward(RewardFunction):
    def __init__(self, env_metadata, district_target, scale=1000.0, **kwargs):
        super().__init__(env_metadata)
        self._target = list(district_target)
        self._scale  = float(scale)
        self._step   = 0

    def calculate(self, observations):
        tgt  = self._target[min(self._step, len(self._target) - 1)]
        load = sum(o.get('net_electricity_consumption', 0.0) for o in observations)
        r    = -((load - tgt) ** 2) * self._scale
        self._step += 1
        n = len(observations)
        return [r / n] * n

    def reset(self):
        self._step = 0

print('TrackingReward defined.')


TrackingReward defined.


## 3. W&B Logging

In [4]:
def log_to_wandb(controller_name, district_load, kpis=None, building_temps=None):
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=controller_name, reinit=True,
                     settings=wandb.Settings(console='off'))
    if kpis is not None:
        wandb.log({'NMBE [%]': float(kpis['NMBE [%]']),
                   'CV-RMSE [%]': float(kpis['CV-RMSE [%]']),
                   'Temp Comfort violation [%]': float(kpis['Temp Comfort violation [%]'])})
    wandb.define_metric('hour')
    wandb.define_metric('district_load', step_metric='hour')
    for t, v in enumerate(district_load):
        wandb.log({'hour': t, 'district_load': float(v)})
    if building_temps is not None:
        T = building_temps.shape[0]
        fig = go.Figure()
        fig.add_hrect(y0=22, y1=26, fillcolor='green', opacity=0.12, line_width=0,
                      annotation_text='Comfort [22-26C]', annotation_position='top right')
        fig.add_hline(y=22, line_dash='dash', line_color='red', line_width=1)
        fig.add_hline(y=26, line_dash='dash', line_color='red', line_width=1)
        for b in range(building_temps.shape[1]):
            fig.add_trace(go.Scatter(x=list(range(T)), y=building_temps[:, b], mode='lines',
                                     line=dict(color='rgba(70,130,180,0.2)', width=1), showlegend=False))
        mean_t = building_temps.mean(axis=1)
        fig.add_trace(go.Scatter(x=list(range(T)), y=mean_t, mode='lines',
                                 line=dict(color='steelblue', width=3), name='Mean temp'))
        fig.update_layout(title=f'{controller_name} - Indoor Temperature',
                          xaxis_title='Hour', yaxis_title='Temp [C]')
        wandb.log({'temperature_plot': wandb.Plotly(fig)})
    wandb.finish()
    print(f'Uploaded to W&B: {controller_name}')

print('log_to_wandb defined.')


log_to_wandb defined.


## 4. UpperSAC

In [5]:
class UpperSAC:
    def __init__(self, obs_dim, act_dim=1, hidden=None, lr=3e-4,
                 buffer_capacity=72000, batch_size=128,
                 gamma=0.99, tau=0.005, alpha=0.2, standardize_after=512):
        hidden = hidden or [64, 64]
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        import gymnasium as gym
        self.action_space = gym.spaces.Box(
            low=np.full(act_dim, -1.0), high=np.full(act_dim, 1.0), dtype=np.float32)
        self.q1  = SoftQNetwork(obs_dim, act_dim, hidden).to(self.device)
        self.q2  = SoftQNetwork(obs_dim, act_dim, hidden).to(self.device)
        self.tq1 = SoftQNetwork(obs_dim, act_dim, hidden).to(self.device)
        self.tq2 = SoftQNetwork(obs_dim, act_dim, hidden).to(self.device)
        self.tq1.load_state_dict(self.q1.state_dict())
        self.tq2.load_state_dict(self.q2.state_dict())
        self.policy = PolicyNetwork(obs_dim, act_dim, self.action_space, 1.0, hidden).to(self.device)
        self.q1_opt = optim.Adam(self.q1.parameters(), lr=lr)
        self.q2_opt = optim.Adam(self.q2.parameters(), lr=lr)
        self.p_opt  = optim.Adam(self.policy.parameters(), lr=lr)
        self.buf  = ReplayBuffer(buffer_capacity)
        self.crit = nn.SmoothL1Loss()
        self.batch_size = batch_size; self.gamma = gamma
        self.tau = tau; self.alpha = alpha
        self.standardize_after = standardize_after
        self.norm_mean = self.norm_std = None; self._normalized = False

    def _norm(self, obs):
        if self.norm_mean is None: return obs
        return (obs - self.norm_mean) / (self.norm_std + 1e-5)

    def act(self, obs, deterministic=False):
        # no_grad prevents building an unused computation graph on every inference call
        with torch.no_grad():
            o = torch.FloatTensor(self._norm(obs)).unsqueeze(0).to(self.device)
            result = self.policy.sample(o)
            a = result[2] if deterministic else result[0]
        return a.detach().cpu().numpy()[0]

    def update(self, obs, action, reward, next_obs, done):
        action = np.atleast_1d(np.array(action, dtype=np.float32))
        # Normalize before pushing once normalization is active so the buffer
        # never contains a mix of raw and normalized observations.
        push_obs  = self._norm(obs)      if self._normalized else obs
        push_next = self._norm(next_obs) if self._normalized else next_obs
        self.buf.push(push_obs, action, reward, push_next, done)
        if len(self.buf.buffer) < self.batch_size: return
        if not self._normalized and len(self.buf.buffer) >= self.standardize_after:
            X = np.array([j[0] for j in self.buf.buffer], dtype=float)
            self.norm_mean = np.nanmean(X, axis=0)
            self.norm_std  = np.nanstd(X, axis=0) + 1e-5
            self.buf.buffer = [(self._norm(o), a, r, self._norm(n), d)
                               for o, a, r, n, d in self.buf.buffer]
            self._normalized = True
        o, a, r, n, d = self.buf.sample(self.batch_size)
        F = torch.FloatTensor
        o=F(o).to(self.device); n=F(n).to(self.device); a=F(a).to(self.device)
        r=F(r).unsqueeze(1).to(self.device); d=F(d).unsqueeze(1).to(self.device)
        with torch.no_grad():
            na, nlp, _ = self.policy.sample(n)
            qt = torch.min(self.tq1(n,na), self.tq2(n,na)) - self.alpha*nlp
            qt = r + (1 - d)*self.gamma*qt
        for q, opt in [(self.q1, self.q1_opt), (self.q2, self.q2_opt)]:
            loss = self.crit(q(o,a), qt); opt.zero_grad(); loss.backward(); opt.step()
        na2, lp2, _ = self.policy.sample(o)
        pl = (self.alpha*lp2 - torch.min(self.q1(o,na2), self.q2(o,na2))).mean()
        self.p_opt.zero_grad(); pl.backward(); self.p_opt.step()
        for tq, q in [(self.tq1,self.q1),(self.tq2,self.q2)]:
            for tp, p in zip(tq.parameters(), q.parameters()):
                tp.data.copy_(tp.data*(1-self.tau) + p.data*self.tau)

print('UpperSAC defined.')


UpperSAC defined.


## 5. DecentralizedFESAC (for loading frozen lower agents)

In [6]:
class DecentralizedFESAC:
    FE_OBS_DIM = 9

    def __init__(self, env, act_dim=1):
        self.env     = env
        self.act_dim = act_dim
        # Filter out non-list entries: CityLearn appends a bare string
        # ('district_load') to observation_names which would break FeatureEngineer.
        obs_names    = [n for n in env.observation_names if isinstance(n, list)]
        self.engineers = [FeatureEngineer(names) for names in obs_names]
        self.agents    = [UpperSAC(obs_dim=self.FE_OBS_DIM, act_dim=act_dim)
                          for _ in obs_names]

    def _transform(self, raw_obs_list):
        return [np.array(eng.process_for_ml(obs), dtype=np.float32)
                for eng, obs in zip(self.engineers, raw_obs_list)]

    def _reset_engineers(self, initial_obs_list=None):
        # Pre-fills temp history from initial obs so rolling_temp_3h is a proper
        # 3-step average from step 0, not a 1-sample average for the first two steps.
        for i, eng in enumerate(self.engineers):
            eng.temp_history.clear()
            if initial_obs_list is not None and eng.outdoor_temp_idx is not None:
                init_t = float(initial_obs_list[i][eng.outdoor_temp_idx])
                eng.temp_history.extend([init_t, init_t])  # 2 priming values

    def _clip_actions(self, actions):
        # battery: [-1, 1]; cooling_device: [0, 1] (negative = physically impossible)
        clipped = []
        for a in actions:
            a = list(np.atleast_1d(a))
            a[0] = float(np.clip(a[0], -1.0, 1.0))
            if len(a) > 1:
                a[1] = float(np.clip(a[1], 0.0, 1.0))
            clipped.append(a)
        return clipped

    def train(self, n_episodes=N_EPISODES, log_every=10):
        t0 = time.time(); ep_rewards = []
        for ep in range(n_episodes):
            raw_obs, _ = self.env.reset()
            self._reset_engineers(raw_obs)
            feat = self._transform(raw_obs)
            ep_r = 0.0
            while not self.env.terminated:
                actions = [ag.act(f, deterministic=False)
                           for ag, f in zip(self.agents, feat)]
                actions = self._clip_actions(actions)
                raw_next, rewards, terminated, truncated, _ = self.env.step(actions)
                feat_next = self._transform(raw_next)
                for i, (ag, f, a, r, fn) in enumerate(
                        zip(self.agents, feat, actions, rewards, feat_next)):
                    ag.update(f, a, float(r), fn, terminated)
                feat = feat_next
                ep_r += float(np.sum(rewards))
            ep_rewards.append(ep_r)
            if (ep+1) % log_every == 0 or ep == 0:
                print(f'  ep {ep+1:3d}/{n_episodes}  reward={ep_r:9.1f}  '
                      f'elapsed={(time.time()-t0)/60:.1f} min')
        print(f'Training done in {(time.time()-t0)/60:.1f} min')
        return ep_rewards

    def predict(self, raw_obs_list, deterministic=True):
        feat = self._transform(raw_obs_list)
        return [ag.act(f, deterministic=deterministic)
                for ag, f in zip(self.agents, feat)]

    def evaluate(self, eval_env):
        raw_obs, _ = eval_env.reset()
        self._reset_engineers(raw_obs)
        feat = self._transform(raw_obs)
        while not eval_env.terminated:
            actions = [ag.act(f, deterministic=True)
                       for ag, f in zip(self.agents, feat)]
            actions = self._clip_actions(actions)
            raw_next, _, _, _, _ = eval_env.step(actions)
            feat = self._transform(raw_next)

    def save(self, directory):
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        for i, ag in enumerate(self.agents):
            torch.save(ag.policy.state_dict(),    directory / f'policy_{i}.pt')
            torch.save(ag.q1.state_dict(),        directory / f'q1_{i}.pt')
            torch.save(ag.q2.state_dict(),        directory / f'q2_{i}.pt')
            np.save(directory / f'norm_mean_{i}.npy',
                    ag.norm_mean if ag.norm_mean is not None else np.array([]))
            np.save(directory / f'norm_std_{i}.npy',
                    ag.norm_std  if ag.norm_std  is not None else np.array([]))
        print(f'Saved {len(self.agents)} building agents to {directory}')

    @classmethod
    def load(cls, env, directory, act_dim=1):
        obj = cls(env, act_dim=act_dim)
        directory = Path(directory)
        for i, ag in enumerate(obj.agents):
            ag.policy.load_state_dict(torch.load(directory / f'policy_{i}.pt',
                                                   map_location=ag.device))
            ag.q1.load_state_dict(torch.load(directory / f'q1_{i}.pt',
                                              map_location=ag.device))
            ag.q2.load_state_dict(torch.load(directory / f'q2_{i}.pt',
                                              map_location=ag.device))
            nm = np.load(directory / f'norm_mean_{i}.npy')
            ns = np.load(directory / f'norm_std_{i}.npy')
            ag.norm_mean = nm if nm.size else None
            ag.norm_std  = ns if ns.size else None
            ag._normalized = ag.norm_mean is not None
        print(f'Loaded {len(obj.agents)} building agents from {directory}')
        return obj

print('DecentralizedFESAC defined  (9-dim FeatureEngineer obs per building).')


DecentralizedFESAC defined  (9-dim FeatureEngineer obs per building).


## 6. Hierarchical Environment

In [7]:
class HierarchicalEnvBase:
    OBS_DIM = 9
    DELTA_SCALE  = 0.35   # upper action ±1 → ±0.35 on each battery
    EMA_ALPHA    = 0.4    # weight on new action (0=frozen, 1=no smoothing)
    ACTION_PENALTY = 0.05 # penalises large corrections in reward

    def __init__(self, citylearn_env, lower_agent, target):
        self._env    = citylearn_env
        self._lower  = lower_agent
        self._target = target
        self._step   = 0
        self._prev_load   = float(target[0])
        self._smooth_act  = 0.0   # EMA state
        self._lower_obs   = None
        self.terminated   = False

    def _mean_soc(self):
        socs = []
        for b in self._env.unwrapped.buildings:
            s = b.electrical_storage.soc
            socs.append(float(s[-1]) if len(s) > 0 else 0.5)
        return float(np.mean(socs))

    def _upper_obs(self):
        h   = self._step % 24
        t   = min(self._step, len(self._target) - 1)
        tgt    = self._target[t]
        tgt_6  = self._target[min(t + 6,  len(self._target) - 1)]
        tgt_12 = self._target[min(t + 12, len(self._target) - 1)]
        return np.array([
            np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24),
            self._prev_load / 30.0, tgt / 30.0,
            (self._prev_load - tgt) / 30.0,
            self._mean_soc(),
            tgt_6 / 30.0, tgt_12 / 30.0,
            (tgt_6 - tgt) / 10.0,
        ], dtype=np.float32)

    def reset(self):
        self._lower_obs, _ = self._env.reset()
        self._lower._reset_engineers(self._lower_obs)  # pass initial obs for temp warmup
        self._step = 0
        self._prev_load  = float(self._target[0])
        self._smooth_act = 0.0
        self.terminated  = False
        return self._upper_obs(), {}

    def _apply_correction(self, delta_a):
        lower_actions = self._lower.predict(self._lower_obs, deterministic=True)
        corrected = []
        for a in lower_actions:
            al = list(np.atleast_1d(a))
            al[0] = float(np.clip(al[0] + delta_a, -1.0, 1.0))
            corrected.append(al)
        try:
            self._lower_obs, _, term, _, _ = self._env.step(corrected)
        except AssertionError:
            self._lower_obs, _, term, _, _ = self._env.step(
                [list(np.atleast_1d(a)) for a in lower_actions])
        return term

    def step(self, upper_action):
        # Fix 1: scale raw action down so upper agent can't swing all 25 batteries at once
        raw = float(np.clip(upper_action[0], -1.0, 1.0)) * self.DELTA_SCALE

        # Fix 2: EMA smoothing — prevents sudden action reversals
        delta_a = self.EMA_ALPHA * raw + (1 - self.EMA_ALPHA) * self._smooth_act
        self._smooth_act = delta_a

        term = self._apply_correction(delta_a)
        self._prev_load = float(sum(
            b.net_electricity_consumption[-1] for b in self._env.unwrapped.buildings))
        tgt = self._target[min(self._step, len(self._target) - 1)]

        # Fix 3: action penalty discourages large corrections
        reward = -((self._prev_load - tgt) ** 2) * 0.01 - self.ACTION_PENALTY * delta_a ** 2

        self._step += 1
        self.terminated = term
        obs = self._upper_obs() if not term else np.zeros(self.OBS_DIM, dtype=np.float32)
        return obs, reward, term, False, {}

    @property
    def net_electricity_consumption(self):
        return self._env.net_electricity_consumption

print('HierarchicalEnvBase defined  (delta_scale=0.35, EMA=0.4, action_penalty=0.05).')

HierarchicalEnvBase defined  (delta_scale=0.35, EMA=0.4, action_penalty=0.05).


## 7. Load Frozen Lower Agents

In [8]:
FROZEN_MODEL = MODELS_DIR / 'Decentralized-SAC-Battery'
assert FROZEN_MODEL.exists(), (
    f'Frozen model not found: {FROZEN_MODEL}\n'
    'Run decentralized_sac_battery.ipynb first.')

def make_frozen_env():
    with open(SCHEMA_PATH) as f:
        schema = json.load(f)
    schema = copy.deepcopy(schema)
    for bid in schema['buildings']:
        schema['buildings'][bid]['inactive_actions'] = ['cooling_device']
    env = CityLearnEnv(
        schema=schema,
        root_directory=str(DATASET_DIR),
        central_agent=False,
        reward_function=TrackingReward,
        reward_function_kwargs={'district_target': target_list, 'scale': TRACKING_SCALE},
    )
    env.reset()
    agent = DecentralizedFESAC.load(env, FROZEN_MODEL, act_dim=1)
    return env, agent

print('make_frozen_env defined. Frozen model path:', FROZEN_MODEL)


make_frozen_env defined. Frozen model path: c:\Users\20213697\OneDrive - TU Eindhoven\2025-26\AIES\Team Internship\annex96_common_exercise_1\notebooks\trained_models\Decentralized-SAC-Battery


## 8. Train Upper-Level SAC

In [9]:
def train_upper_level(EnvClass, act_dim=1, n_episodes=N_EPISODES):
    t0 = time.time()
    dec_env, frozen_agent = make_frozen_env()
    hier_env  = EnvClass(dec_env, frozen_agent, district_target)
    upper_sac = UpperSAC(EnvClass.OBS_DIM, act_dim=act_dim)
    ep_rewards = []
    print(f'Training upper-level SAC ({EnvClass.__name__}, {n_episodes} episodes)...')
    for ep in range(n_episodes):
        obs, _ = hier_env.reset()
        ep_r   = 0.0
        while not hier_env.terminated:
            action   = upper_sac.act(obs, deterministic=False)
            next_obs, reward, done, _, _ = hier_env.step(action)
            upper_sac.update(obs, action, reward, next_obs, done)
            obs   = next_obs; ep_r += reward
        ep_rewards.append(ep_r)
        if (ep + 1) % 10 == 0 or ep == 0:
            print(f'  ep {ep+1:3d}/{n_episodes}  reward={ep_r:9.1f}  '
                  f'elapsed={(time.time()-t0)/60:.1f} min')
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    return upper_sac, np.array(ep_rewards)

print('train_upper_level defined.')


train_upper_level defined.


In [10]:
upper_sac, ep_rewards = train_upper_level(HierarchicalEnvBase, act_dim=1)
HierEnvClass = HierarchicalEnvBase


Loaded 25 building agents from c:\Users\20213697\OneDrive - TU Eindhoven\2025-26\AIES\Team Internship\annex96_common_exercise_1\notebooks\trained_models\Decentralized-SAC-Battery
Training upper-level SAC (HierarchicalEnvBase, 100 episodes)...
  ep   1/100  reward=  -4465.1  elapsed=2.6 min
  ep  10/100  reward=   -496.0  elapsed=22.8 min
  ep  20/100  reward=   -465.7  elapsed=44.5 min
  ep  30/100  reward=   -473.7  elapsed=66.3 min
  ep  40/100  reward=   -432.2  elapsed=87.9 min
  ep  50/100  reward=   -425.9  elapsed=110.1 min
  ep  60/100  reward=   -396.2  elapsed=131.8 min
  ep  70/100  reward=   -420.1  elapsed=153.4 min
  ep  80/100  reward=   -414.7  elapsed=175.5 min
  ep  90/100  reward=   -378.6  elapsed=199.3 min
  ep 100/100  reward=   -393.9  elapsed=221.8 min
Done in 221.8 min


In [ ]:
# Save upper SAC immediately after training so evaluation survives a kernel restart
save_dir = MODELS_DIR / CONTROLLER_NAME
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(upper_sac.policy.state_dict(), save_dir / 'upper_policy.pt')
np.save(save_dir / 'upper_norm_mean.npy',
        upper_sac.norm_mean if upper_sac.norm_mean is not None else np.array([]))
np.save(save_dir / 'upper_norm_std.npy',
        upper_sac.norm_std  if upper_sac.norm_std  is not None else np.array([]))
np.save(save_dir / 'episode_rewards.npy', np.array(ep_rewards))
print(f'Upper SAC saved to {save_dir}')

## 9. Evaluate & Upload

In [ ]:
# Auto-reload upper SAC from disk if kernel was restarted after training
if 'upper_sac' not in vars():
    _act = getattr(HierEnvClass, 'ACT_DIM', 1)
    upper_sac = UpperSAC(HierEnvClass.OBS_DIM, act_dim=_act)
    _sd = MODELS_DIR / CONTROLLER_NAME
    upper_sac.policy.load_state_dict(
        torch.load(_sd / 'upper_policy.pt', map_location=upper_sac.device))
    _nm = np.load(_sd / 'upper_norm_mean.npy')
    _ns = np.load(_sd / 'upper_norm_std.npy')
    upper_sac.norm_mean = _nm if _nm.size else None
    upper_sac.norm_std  = _ns if _ns.size else None
    upper_sac._normalized = upper_sac.norm_mean is not None
    print('Reloaded upper SAC from disk.')
# Deterministic evaluation
dec_env, frozen_agent = make_frozen_env()
eval_hier = HierEnvClass(dec_env, frozen_agent, district_target)
obs, _ = eval_hier.reset()
while not eval_hier.terminated:
    action = upper_sac.act(obs, deterministic=True)
    obs, _, _, _, _ = eval_hier.step(action)

district_load  = np.array(eval_hier.net_electricity_consumption)
building_temps = np.array([
    b.energy_simulation.indoor_dry_bulb_temperature[:720]
    for b in dec_env.buildings
]).T  # (720, 25)

kpis = compute_kpis(district_target, district_load, building_temps)
print('KPIs:', kpis)
log_to_wandb(CONTROLLER_NAME, district_load, kpis, building_temps)


Loaded 25 building agents from c:\Users\20213697\OneDrive - TU Eindhoven\2025-26\AIES\Team Internship\annex96_common_exercise_1\notebooks\trained_models\Decentralized-SAC-Battery
KPIs: {'NMBE [%]': 5.084, 'CV-RMSE [%]': 34.533, 'Temp Comfort violation [%]': 27.75}
Uploaded to W&B: Hierarchical-Hier2-Forecast


: 